# Kalman v2 — Panel Fit, Forecast & Portfolio Replay

Notebook form of two workflows that are otherwise CLI-only:

| Part | Script | What it does |
|---|---|---|
| **A** | `pymc_kalman_filter_pt_v2.py` | Loads the trail panel, audits it, fits the marginalised MvStudentT model, screens it, sizes a CVaR risk book, writes `07_forecast_handoff_v2.nc` and exports the analytics frames |
| **B** | `kalman_portfolio.py` | Picks that handoff up and replays everything downstream: forward simulation, the two prior sweeps, three ranking arms, the recommendation layer |

**Why a notebook.** Both scripts are one-shot. The objects that make the model
arguable — `panel`, `idata`, `screen_draws`, `ForecastDraws`, the sized
`Portfolio` — exist only inside `main()`, and the 33 figure panels are invoked
once, at the end, as a best-effort `render_run` / `render_replay` call. Here each
stage is a cell and **each figure sits at the step that produces its input**.

**Nothing below is reimplemented.** Every cell calls the module function that
already owns the stage; the notebook is orchestration and commentary only. Where
this notebook departs from the scripts it is stated in the cell.

---

### Two things to know before running

**1. `FIT_MODEL` decides how long this takes.** With it `True` the notebook
samples (tens of minutes). With it `False` Part A is skipped entirely and Part B
replays an existing handoff off disk in seconds — which is the point of the
handoff, and the right mode for iterating on the decision layer.

**2. Environment is set *before* import, not here.** `PYTENSOR_FLAGS` must be
normalised before PyTensor is first imported, so importing
`probabilistic_ml_model` runs `force_python_vm()` as a side effect. `set_env.ps1`
owns the full setup; the C backend is off project-wide unless
`PML_ENABLE_PYTENSOR_C=1` is set before import. For a **redirected** run also set
`PYTHONIOENCODING=utf-8` in the shell — several stages print `ρ` and `κ`, and
Windows stdout falls back to cp1252 when piped.

## §0 — Imports & configuration

Two frozen config objects, and the split between them is load-bearing:

| Config | Owns |
|---|---|
| `KalmanModelConfig` | Everything that changes the **posterior** — lookbacks, group effects, likelihood, the variance split, the scale model |
| `KalmanRunConfigV2` | Everything that changes only the **run** — NUTS budget, gates, export paths, which optional stages fire |

Both are frozen: override with `dataclasses.replace`, never by mutation.

`GateReport` is threaded through every stage. Fit quality here is *self-reported
by code*, never scraped from console output — a stage that measures something
adds a `GateResult` and the report is rendered at the end of each Part.

In [ ]:
import logging
from dataclasses import replace

import numpy as np
import pandas as pd
import pymc as pm

# Both workflows are the single source of truth for their own stages — import
# them, never re-define a stage in a cell.
import pymc_kalman_filter_pt_v2 as kf2
import kalman_portfolio as kp

from pymc_kalman_filter_pt_v2 import (
    # --- config / gates ---
    KalmanRunConfigV2, GateReport, GATE_CATALOGUE,
    # --- §1 data -> §4 panel -> §4b audit ---
    load_kalman_frame, select_drift_features_v2, prepare_panel,
    run_panel_diagnostics,
    # --- §6 prior -> §7 fit -> §8 ppc -> §9 diagnostics ---
    run_runtime_estimate, run_prior_predictive, sample_posterior,
    run_diagnostics, run_posterior_predictive,
    # --- §9b comparison ---
    compare_arms_fast, run_model_comparison,
    # --- §10 screen -> handoff -> §10b risk -> §15 forecast ---
    run_screen, write_forecast_handoff, run_risk_book, run_forecast_layer,
    # --- §10c export / §14 summary ---
    apply_out_of_support, export_analytics, summarise,
)
from probabilistic_ml_model.pymc_models.KalmanFilterModel_v2 import (
    KalmanModelConfig, build_kalman_pt_model_v2,
    # Group-effect presets for the §9b arms and the §0b overrides below.
    GROUP_EFFECTS_NESTED_GEO, GROUP_EFFECTS_NESTED_FULL, GROUP_EFFECTS_STYLED,
)
from probabilistic_ml_model.pymc_models.PortfolioOptimizationModel import (
    RANKING_RULES, PORTFOLIO_OBJECTIVES, DEFAULT_RANKING_RULE,
)
from probabilistic_ml_model.pymc_models.KalmanForecast import ForecastConfig

# The two figure layers, now inside the visualizations package beside the shared
# primitives they render through (moved 2026-08-28).
from probabilistic_ml_model.visualizations import kalman_viz_v2 as viz
from probabilistic_ml_model.visualizations import kalman_portfolio_viz as pviz
from probabilistic_ml_model.visualizations.kalman_shared import (
    set_export_section, get_export_state,
)

model_cfg = KalmanModelConfig()
run_cfg = KalmanRunConfigV2.from_env()

logging.basicConfig(level=getattr(logging, run_cfg.log_level.upper(), logging.INFO),
                    format='%(asctime)s %(levelname)-7s %(name)s: %(message)s')

print('Orchestrating :', kf2.__file__)
print('Replaying     :', kp.__file__)
print('Results root  :', run_cfg.results_path)
model_cfg

### §0b — Run toggles

| Toggle | Effect |
|---|---|
| `FIT_MODEL` | `False` skips **all of Part A** and replays an existing handoff. Part B then reads the screen, MC summary and diagnostics from the v2 results tree instead of memory. |
| `EXPORT_ARTIFACTS` | Persist figures and tables under the v2 results root. |
| `WRITE_ANALYTICS` | `True` **DROPs and RECREATEs** `analytics.kalman_filtered_price_targets_v2`, the GEIB dashboard's only source. |

**`WRITE_ANALYTICS` ships `False`, unlike the v4 notebook.** A destructive
schema write is not something a notebook should do as a side effect of being run
top-to-bottom, and a re-export must be paired with a dashboard deploy. Production
refresh belongs to `scripts/export_kalman_analytics.py`.

`viz.install` does three things in one call — points the shared figure layer's
config resolver at `run_cfg`, applies the dark template, and enables artifact
export — so it replaces the bare `enable_artifact_export()` the v1 notebooks
call. It is invoked **once**, here; every cell below then just tags its section.

In [ ]:
# --- notebook toggles ---------------------------------------------------------
FIT_MODEL = True          # False -> skip Part A, replay an existing handoff
EXPORT_ARTIFACTS = True   # persist figures + tables under the v2 results root
WRITE_ANALYTICS = False   # True DROPs and RECREATEs the GEIB source table

# --- model overrides (uncomment as needed) ------------------------------------
# model_cfg = replace(model_cfg, group_parents={})                   # crossed (shipped)
# model_cfg = replace(model_cfg, group_effects=GROUP_EFFECTS_NESTED_GEO)
# model_cfg = replace(model_cfg, likelihood='normal')                # contrast only

# --- run overrides ------------------------------------------------------------
# run_cfg = replace(run_cfg, draws=200, tune=200, chains=2)          # smoke fit
# run_cfg = replace(run_cfg, enable_model_comparison=True)           # §9b, ~3x cost
# run_cfg = replace(run_cfg, forecast_factor_share=0.0)              # independent shocks

# Chains run SEQUENTIALLY in a notebook. nutpie's parallel native workers crash an
# IDE-managed Jupyter kernel on Windows -- an uncatchable native crash that surfaces
# only as "Connection to IDE-Managed Server is lost". Wall-clock only: chains, seeds
# and the posterior are identical to the script path.
run_cfg = replace(run_cfg, cores=1, write_analytics=WRITE_ANALYTICS)

viz.install(run_cfg, enable=EXPORT_ARTIFACTS)
report = GateReport()

_st = get_export_state()
print(f'artifact export : {"ON -> " + str(_st.root) if EXPORT_ARTIFACTS else "OFF (display only)"}')
print(f'run_id          : {_st.run_id}  (started {_st.started_at:%Y-%m-%d %H:%M:%S} UTC)')
print(f'mode            : {"FIT + REPLAY" if FIT_MODEL else "REPLAY ONLY (Part A skipped)"}')
print(f'NUTS budget     : draws={run_cfg.draws} tune={run_cfg.tune} '
      f'chains={run_cfg.chains} cores={run_cfg.cores} '
      f'target_accept={run_cfg.target_accept}  sampler={run_cfg.nuts_sampler}')
print(f'likelihood      : {model_cfg.likelihood} (nu floor {model_cfg.nu_floor})')
print(f'group effects   : {model_cfg.group_effects}')
print(f'                  parents={model_cfg.group_parents or "None -> CROSSED (shipped)"}')
print(f'lookbacks       : {model_cfg.lookbacks}  (T={model_cfg.n_time})')
print(f'forecast layer  : enabled={run_cfg.enable_forecast_layer} '
      f'factor_share={run_cfg.forecast_factor_share} '
      f'scenarios={run_cfg.forecast_scenarios}')
print(f'write_analytics : {run_cfg.write_analytics}')

---

# Part A — The fit

Mirrors `pymc_kalman_filter_pt_v2.main()` stage for stage. **The order is the
script's, not the section numbers'**, and two places matter:

- **§9 diagnostics runs before §8 PPC.** Convergence is a precondition for
  reading a predictive check; a PPC on a fit that did not converge describes
  nothing.
- **The handoff is written after §10, not after §7.** `screen_draws.eu` is the
  *shrunk* decision latent. A handoff carrying the raw one would replay a
  different model from the run that produced it, and every gate would still pass.

Every cell below is wrapped in `if FIT_MODEL:` so the whole Part can be skipped.

## §1 — Data

`load_kalman_frame` queries `pml.mv_pymc_kalman_pt_v2` — the v2 materialized
view, which is `SELECT b.*` over its v1 parent plus the correlated response trail
`feat_log_uplift_{now,1w,1m,3m,6m,1y}`, the per-lookback analyst counts that drive
the per-cell measurement scale, and the split EPS block.

Filtered by `min_next_earnings`, `min_report_date` and `min_trail_obs` from the
run config — a name with one trail observation cannot inform a decay kernel.

In [ ]:
if FIT_MODEL:
    set_export_section('01_data')

    frame = load_kalman_frame(run_cfg)
    print(f'{frame.shape[0]:,} ISINs x {frame.shape[1]} columns')
    display(frame[['isin', 'ticker', 'name', 'sector', 'trading_region',
                   'n_trail_obs', 'n_analysts']].head())

## §3 — Drift features

**The catalogue decides which columns exist; this decides which belong in the
state-transition mean.** They are different questions, and conflating them once
cost v1 a broken coverage check: flipping a `pymc_role` to `'excluded'` in SQL
drops the row from `vw_pymc_feature_catalogue` while the MV still emits the
column, and `assert_pymc_catalogue_coverage()` then raises
`MISSING_FROM_CATALOGUE`. **Exclusions live in Python, always.**

Four tests in order — two *named* (a column somebody decided about) and two
*measured* (coverage and signal against the response), so a feature that stops
carrying information drops out on its own rather than waiting to be noticed.

In [ ]:
if FIT_MODEL:
    set_export_section('03_features')

    drift_names = select_drift_features_v2(frame)
    print(f'{len(drift_names)} drift features admitted:')
    for _n in drift_names:
        print('   ', _n)

## §4 — Panel preparation

Builds the `(n_isin, T)` response matrix `Y` on the genuine `(isin, time)` grid,
standardises the drift design matrix, resolves the hierarchy indices, and carries
the fit-time moments (`response_mean` / `response_std`) that make the §10
de-standardisation exact **by construction** rather than by a second calculation
that can drift.

Also rotates the collinear price-target-history family
(`orthogonalise_pt_history`), recording the rotation so a coefficient can be read
back onto its source columns.

In [ ]:
if FIT_MODEL:
    set_export_section('04_panel')

    panel = prepare_panel(frame, model_cfg, run_cfg, drift_names=drift_names)
    print(f'Y shape (isin, time) : {panel.Y.shape}')
    print(f'time grid (days)     : {panel.time_days.tolist()}')
    print(f'observed cells       : {int(panel.observed_mask.sum()):,} '
          f'of {panel.Y.size:,} ({panel.observed_mask.mean():.1%})')
    print(f'drift matrix         : {panel.X_drift.shape}')
    print(f'de-standardisation   : mean={panel.response_mean:.6f} '
          f'std={panel.response_std:.6f}')
    print(f'hierarchy levels     : {list(panel.coord_uniques)}')
    panel.frame.head()

## §4b — Panel information audit

**The stage that decides whether fitting is worth it at all**, and the reason
`--dry-run` exists: it costs seconds and answers the question a 45-minute fit
would answer expensively.

Two blocking gates:

- **`panel_t_eff`** — the *effective* number of time points. Four observations of
  a trail that decorrelates slowly are not four independent observations; `T_eff`
  is the equicorrelated-block equivalent, and below `gate_t_eff_min` the panel
  cannot identify what the model asks of it.
- **`panel_kernel_fit`** — how well `rho_inf + (1-rho_inf)·exp(-Δ/ℓ)` describes
  the measured trail correlation. The OU kernel is the model's entire theory of
  how a price target goes stale; if it does not fit the data, nothing downstream
  is interpretable.

The two panels draw exactly this: the measured decay against the fitted kernel,
and the information the grid actually carries.

In [ ]:
if FIT_MODEL:
    set_export_section('04b_audit')

    audit = run_panel_diagnostics(panel, run_cfg, report)
    # `run_panel_diagnostics` does `out.update(kern)`, so the kernel keys are FLAT
    # on the audit dict rather than nested under a 'kernel' entry.
    print(f"T_eff        : {audit['t_eff']:.3f}  (gate >= {run_cfg.gate_t_eff_min})")
    print(f"kernel       : rho_inf={audit['rho_inf']:.4f} "
          f"ell={audit['ell_days']:.1f}d  half-life={audit['half_life_days']:.0f}d "
          f"rmse={audit['rmse']:.4f}")
    print(f"implied split: {audit['rho_inf'] * 100:.0f}% permanent level, "
          f"{(1 - audit['rho_inf']) * 100:.0f}% decaying")
    print(f"per-step cov : "
          f"{np.round(audit['per_step_coverage'], 3).tolist()}")

    viz.plot_panel_audit(panel, audit)
    viz.plot_decay_ladder(panel, audit)

## §5b — Build the model

`build_kalman_pt_model_v2` assembles the marginalised MvStudentT trail model: a
per-name regression mean `mu_reg` over the drift matrix plus group effects, and a
covariance combining a permanent level, an OU-decaying state and per-cell
measurement noise, with the three shares drawn from a Dirichlet.

**Group effects are crossed by default** (`group_parents=None`), and that
identity is what makes a nested arm a one-change contrast:
`build_group_effect_terms` is the single definition, used by the full model and
by the Max-and-Smooth screener alike — so an arm is *screened* as the model it
would be *fitted* as.

**Only the leaf of each chain enters `eta`.** An intermediate level contributes
through its children; adding its own term as well would count it once per level
of depth below it. The builder asserts this rather than trusting it, because the
failure is silent — the model still samples, the mean is just wrong.

In [ ]:
if FIT_MODEL:
    set_export_section('04_panel')

    model = build_kalman_pt_model_v2(panel, model_cfg)
    print(f'{len(model.free_RVs)} free RVs, {len(model.observed_RVs)} observed')
    print('free:', sorted(rv.name for rv in model.free_RVs))
    pm.model_to_graphviz(model)

### §7 gate — runtime estimate

**Measure before committing.** Times a handful of gradient evaluations and
extrapolates to the configured budget. A model that cannot finish is caught here,
in seconds, instead of forty-five minutes into a run that has produced eight
draws. Blocking against `max_runtime_minutes`.

In [ ]:
if FIT_MODEL:
    set_export_section('07_posterior')

    runtime = run_runtime_estimate(model, run_cfg, report)
    for _k, _v in runtime.items():
        print(f'{_k:28s}: {_v}')

## §6 — Prior predictive

Draws from the prior and pushes the implied observations back onto the
**interpretable** scale — log-uplift de-standardised to a percentage — then
compares against the empirical distribution. A prior predictive on the
standardised scale answers nothing: the question is whether the model, before
seeing data, considers a +40% price target plausible and a +4000% one not.

Gated by `prior_scale`.

In [ ]:
if FIT_MODEL:
    set_export_section('06_prior')

    prior_idata = run_prior_predictive(model, panel, run_cfg, report)
    viz.plot_prior_predictive(prior_idata, panel)

## §7 — Posterior inference (NUTS)

`sample_posterior` reads the budget from `run_cfg` and builds its kwargs through
the shared `build_sample_kwargs`, so the notebook inherits the project's sampling
policy rather than a second copy of it.

Two things worth knowing here:

- **`cores=1` was set in §0b**, not passed here — the config is the single place
  the budget lives. Chains run sequentially; the posterior is identical.
- **`log_likelihood` is off**, deliberately and project-wide. It roughly doubles
  the `InferenceData` and no production path reads it, so `az.loo` / `az.compare`
  will raise on this object. §9b attaches it post-hoc on a subsample instead —
  see that cell.

This is the expensive cell.

In [ ]:
if FIT_MODEL:
    set_export_section('07_posterior')

    idata = sample_posterior(model, run_cfg)
    print('divergences :', int(idata.sample_stats['diverging'].sum()))
    # ArviZ 1.x returns an xarray.DataTree, where `groups` is a TUPLE attribute;
    # legacy InferenceData exposes it as a method. Handle both rather than
    # assuming -- this is the compat seam the project's InferenceLike alias exists
    # for, and calling the tuple raises TypeError after the fit has been paid for.
    _groups = getattr(idata, 'groups', None)
    if callable(_groups):
        _groups = _groups()
    elif _groups is None:
        _groups = getattr(idata, 'children', {})
    print('groups      :', list(_groups))
    idata.posterior

## §9 — MCMC diagnostics

**Before the predictive check, not after.** Gated on `r_hat` and `ess_bulk`
(`MIN_ESS_GATE = 400`), and both statistics are *between-chain* — with one chain
they come back `NaN` rather than reassuring.

Six panels:

| Panel | Reads |
|---|---|
| `plot_rhat_ess` | the gate itself, per parameter |
| `plot_trace_worst` | the worst-mixing parameters, so a bad R-hat has a picture |
| `plot_energy` | NUTS energy — marginal vs transition (E-BFMI) |
| `plot_variance_legs` | the Dirichlet split: level / state / observation |
| `plot_sigma_time_calibration` | whether the staleness scale matches measured decay |
| `plot_drift_forest` | the drift coefficients, on the rotated basis |

**Watch `nu`.** If it sits pinned at its 2.5 floor it is absorbing scale
mis-specification, not measuring tail weight — check `sigma_base` before touching
it, and never relax the floor.

In [ ]:
if FIT_MODEL:
    set_export_section('09_diagnostics')

    diagnostics = run_diagnostics(idata, panel, run_cfg, report)

    viz.plot_rhat_ess(diagnostics, run_cfg)
    viz.plot_trace_worst(idata, diagnostics)
    viz.plot_energy(idata)
    viz.plot_variance_legs(idata)
    viz.plot_sigma_time_calibration(idata, panel)
    viz.plot_drift_forest(idata, panel)

    display(diagnostics.sort_values('r_hat', ascending=False).head(12))

## §8 — Posterior predictive

A density overlay alone is not a check — it hides everything a decision depends
on. This stage ships **calibration statistics**: per-`y_series` and per-time
coverage against a target, a t-statistic on the spread, and the decay of the
*residual* correlation.

The last is the sharpest one. `ppc_decay_residual` asks whether the model has
actually absorbed the trail's autocorrelation or merely reproduced it: if the
residuals still decay like the raw response, the correlated likelihood is
decorating rather than explaining.

Gated by `ppc_t_spread`, `ppc_coverage`, `ppc_decay`, `ppc_decay_residual`,
`mean_calibration`, `mean_spread` and `coverage_gradient`.

In [ ]:
if FIT_MODEL:
    set_export_section('08_ppc')

    ppc = run_posterior_predictive(model, idata, panel, run_cfg, model_cfg, report)

    viz.plot_ppc_overlay(ppc, panel, ppc.get('ppc_idata'))
    viz.plot_ppc_calibration(ppc, panel, run_cfg)
    viz.plot_ppc_decay(ppc)

## §9b — Model comparison

Two harnesses, and the difference between them is the point.

**`compare_arms_fast` (Max-and-Smooth) — cheap, and on by default here.**
Screens arms in seconds against **one** baseline fit by turning each name's trail
into a Gaussian pseudo-observation of `mu_reg`. It **ranks arms; it does not
decide them.** `COVARIANCE_FIELDS` refuses any arm that touches what the Max step
froze, and `drift_strict` is refused outright because it changes the design
matrix — the quantity the Max step conditioned on.

**`run_model_comparison` (exact ELPD) — commented out.** It refits every arm and
computes a pointwise `log_likelihood` (~820 MB per arm at full panel size), so
roughly 3x the sampling cost. Confirm a winner here before editing a default.

Arms worth screening: `hierarchy_nested` (shrink a one-name country toward its
bloc rather than toward zero), `hierarchy_geo` (its crossed control — nested
changes the level set *and* the parameterisation, so on its own it cannot say
which of the two paid), `hierarchy_nested_full`, `hierarchy_styled`.

In [ ]:
if FIT_MODEL:
    set_export_section('09b_comparison')

    comparison_fast = compare_arms_fast(
        panel, idata, model_cfg, run_cfg, report,
        arms=('hierarchy_nested', 'hierarchy_geo', 'hierarchy_nested_full'),
    )

    # The EXACT contrast - refits each arm, ~3x sampling cost. Confirm here before
    # moving a default; the fast screen above only ranks.
    # comparison = run_model_comparison(
    #     frame, model_cfg, replace(run_cfg, enable_model_comparison=True), report)

    display(comparison_fast if comparison_fast is not None
            else 'fast comparison returned nothing')

## §10 — Screen

Turns the posterior into a per-ISIN decision frame. Everything here is a
**reduction over draws**, computed in `run_screen` rather than in the model graph.

Two columns to read carefully:

- **`p_upside_pos_cond` is the primary ranking column** — P(risk-adjusted forward
  return > 0). `prob_pos` is reported and never ranked.
- **`kalman_gain` keeps its name for one release but is no longer a gain**;
  `shrink_gain` is the column now carrying that meaning.

The forecast-error shrinkage applied here is **a prior, not an estimate**, and the
distinction is load-bearing: without it the screen reproduces analyst consensus at
Spearman 0.999995. `obs_share` is identified by how much the trail decorrelates
across its shortest gap — which measures how noisily a target is *republished*,
not how far consensus sits from fair value. The panel cannot separate the two, so
re-prioring it does not move it. §P2 sweeps the multiplier instead.

`plot_rank_correlations` measures how far each exported column departs from a
consensus sort; `plot_er_sd_calibration` scores the forward second moment against
realised volatility, which is what gives `tail_risk_vol_floor_k` a measured
rationale rather than a value.

In [ ]:
if FIT_MODEL:
    set_export_section('10_screen')

    screen, screen_draws = run_screen(idata, panel, run_cfg, report)
    print(f'{len(screen):,} names screened')
    print(f'median expected upside : {screen["expected_upside"].median():.2%}')
    print(f'median implied upside  : {screen["implied_upside"].median():.2%}')
    print(f'above consensus        : '
          f'{(screen["expected_upside"] > screen["implied_upside"]).mean():.1%}')

    viz.plot_screen_overview(screen, panel)
    viz.plot_rank_correlations(screen)
    viz.plot_er_sd_calibration(screen, panel)

    screen.head(10)

### §7b — Forecast handoff

Persists the four posterior quantities the forward simulation actually reads — the
standardised latent `mu_std`, its `sigma_std`, `nu` and the OU length scale — plus
the identity block and coordinate maps, stamped with `run_id`, `source_sha` and
`source_dirty`.

**Written here, after §10, and not after §7.** `screen_draws.eu` is the *shrunk*
decision latent; a handoff carrying the raw one would replay a different model from
the run that produced it while every gate still passed.

`source_dirty` is a **fact, not an error** — most runs have an uncommitted tree. It
is what tells a reader the SHA does not fully determine what ran.

This file is what makes Part B replayable in seconds, and it is written before the
forecast layer so a run that fails downstream still leaves behind the artifact that
makes the failure reproducible.

In [ ]:
if FIT_MODEL:
    set_export_section('07_posterior')

    run_id = get_export_state().run_id
    handoff_path = write_forecast_handoff(
        idata, panel, screen, screen_draws, run_cfg, run_id, report
    )
    print('handoff ->', handoff_path)

## §10b — CVaR-aware risk book

Ranks on STARR and sizes with cap-and-spill. All columns are **raw decimals**
(0.25 = +25%); percent scaling happens only at print and plot boundaries.

**Risk units come from the forward-return Monte Carlo**, not from the posterior
upside draws — `cvar05` and `exp_vol` read `screen_draws.pooled_returns`. The
estimation-uncertainty view keeps its own accurate name, `expected_upside_sd`.

**Draws are aligned by ISIN, never by position.** `run_screen` returns the screen
sorted by `expected_upside` while `pooled_returns` stays in `panel.isins` order, so
a positional assignment attributes every risk column to the wrong name — a
permutation a length-only guard cannot see. The identity that catches it is
`exp_vol == er_sd` (both are the pooled sd of the same draws), and
`compute_cvar_aware_book` self-checks it on every call.

In [ ]:
if FIT_MODEL:
    set_export_section('10b_risk')

    risk_book = run_risk_book(idata, panel, screen, run_cfg, draws=screen_draws)
    if risk_book is not None:
        for _k, _v in risk_book.summary.items():
            print(f'{_k:26s}: {_v}')
        viz.plot_risk_book(risk_book, run_cfg)
        display(risk_book.book.head(15))
    else:
        print('risk book unavailable (see the log); §10c falls back to the screen')

## §15 / §15b — Forecast layer & decision layer

**Additive: it reports beside the AR simulator, never in place of it.** The shipped
`er_*` / `cvar05` / `starr` columns and the sized risk book are untouched; §15 emits
its own frames so the two engines can be contrasted on one fit.

The two differ in how they decay — a fitted OU kernel here against a hand-set
`rho = 0.85` there — and in horizon: 365 calendar days against four unitless
periods. Flipping the default is a decision for a realised-return vintage, not for a
stage that can only be scored against the trail it was fitted to.

`factor_share` splits shock variance into a common factor and an idiosyncratic part.
The split is **variance-preserving**, so per-name marginals are invariant and only
the *joint* distribution moves — harmless for the screen, decisive for every
portfolio statistic. At 0 the shocks are cross-sectionally independent, which is
what the AR simulator assumes, and is why a long book can otherwise report a
positive expected shortfall: pooling 25 names averages their idiosyncratic risk to
nearly nothing.

In [ ]:
if FIT_MODEL:
    set_export_section('15_forecast')

    forecast_stage = run_forecast_layer(
        idata, panel, screen, run_cfg, report, draws=screen_draws
    )
    print('stage keys:', sorted(forecast_stage))
    if forecast_stage.get('ergodicity'):
        for _k, _v in forecast_stage['ergodicity'].items():
            print(f'{_k:26s}: {_v:.6f}')
    if isinstance(forecast_stage.get('forecast_summary'), pd.DataFrame):
        display(forecast_stage['forecast_summary'].head())

## §10c — Analytics export

Assembles the canonical frame and writes every export table. This cell reproduces
`main()`'s assembly rather than paraphrasing it, because three steps are each there
for a reason:

- **`drop(columns=['expected_sharpe'])`** — a one-release guard. A stale
  `RiskBookModel` emitting both aliases would rename one onto the other, and the
  export gate then hands `pd.to_numeric` a DataFrame instead of a Series and the
  whole export dies *after* the fit has been paid for.
- **`apply_out_of_support`** runs **before** anything is written, so every consumer
  sees the same guarded values. In v1 it ran after the risk table was persisted,
  which is why that table still carries an `expected_sharpe_ratio` of -2,142.
- **`_attach_identity_frames` joins BY ISIN.** The screen is sorted by
  `expected_upside` while `panel.frame` is in universe order; a positional attach
  hands every name someone else's country.

Three export gates run inside `export_analytics`. `export_finite` is **blocking**:
an unbounded quantity is NULL plus a boolean saying why, because a `float8 Infinity`
poisons every downstream `AVG`.

In [ ]:
if FIT_MODEL:
    set_export_section('10c_analytics')

    kalman_results = (risk_book.analytics.copy() if risk_book is not None
                      else screen.copy())
    kalman_results = kalman_results.drop(columns=['expected_sharpe'], errors='ignore')
    kalman_results = kalman_results.rename(columns={
        'starr': 'reward_to_cvar',
        'cvar05': 'cvar_5pct_kalman',
        'book_weight': 'cvar_book_weight',
        'expected_upside': 'expected_return_kalman',
        'expected_pt': 'price_target_kalman',
        'last_price': 'original_price',
        'observed_pt': 'original_target',
    })
    kalman_results = apply_out_of_support(kalman_results)

    frames = {
        '04_panel_frame_v2': panel.frame,
        '09_diagnostics_v2': diagnostics.reset_index(),
        '10_screen_results_v2': screen,
        kf2._ANALYTICS_TABLE_V2: kalman_results,
    }
    if 'er_mean' in screen.columns:
        frames['10_screen_mc_summary_v2'] = screen[
            ['isin', 'er_mean', 'er_sd', 'er_p05', 'er_p50', 'er_p95', 'mc_prob_pos']
        ]
    if comparison_fast is not None and len(comparison_fast):
        frames['09b_comparison_v2'] = comparison_fast.assign(
            backend=comparison_fast.get('backend', 'max_and_smooth'))
    if risk_book is not None:
        frames['10b_risk_analytics_v2'] = apply_out_of_support(
            risk_book.analytics.drop(columns=['expected_sharpe'], errors='ignore'))
        frames[kf2._RISK_BOOK_KEY] = risk_book.book
    if isinstance(forecast_stage.get('forecast_summary'), pd.DataFrame):
        frames['15_forecast_summary_v2'] = apply_out_of_support(
            forecast_stage['forecast_summary'])
    if isinstance(forecast_stage.get('decision_frame'), pd.DataFrame):
        frames['15b_decision_analytics_v2'] = apply_out_of_support(
            forecast_stage['decision_frame'])

    frames = kf2._attach_identity_frames(frames, panel.frame)
    export_counts = export_analytics(frames, run_cfg, report, run_id=run_id)
    for _k, _v in sorted(export_counts.items()):
        print(f'{_k:34s}: {_v:>8,} rows')

## §14 — Gate report

Fit quality is **self-reported by code**, never scraped from console output. Every
stage above added its `GateResult` to the same `report`; this renders them.

A blocking failure means the export should not be trusted, not that the cell
errored — read the report rather than the absence of a traceback.

In [ ]:
if FIT_MODEL:
    set_export_section('09_gates')

    summarise(report, {
        'run_id': run_id,
        'names': panel.n_isin,
        'T / T_eff': f"{panel.n_time} / {audit['t_eff']:.2f}",
        'median expected upside': f"{screen['expected_upside'].median():.2%}",
        'median implied upside': f"{screen['implied_upside'].median():.2%}",
        'above consensus': f"{(screen['expected_upside'] > screen['implied_upside']).mean():.1%}",
    }, results_path=run_cfg.results_path)

    print(f'\nblocking failures: {len(report.blocking_failures)}   ok={report.ok}')

---

# Part B — The forecast & decision replay

Mirrors `kalman_portfolio.main()`. Everything here runs off the handoff, so it is
**seconds per iteration, no sampling and no database write** — which is the whole
reason it is a separate workflow. The layers below are governed by two priors the
panel cannot identify (`forecast_error_multiplier`, `forecast_factor_share`) and by
a choice of ranking rule that two shipped candidates have already failed. Every one
of those questions is answered by running the same fit many times with one thing
changed; inside the fit that costs a NUTS run each, here it costs seconds — the
difference between a sensitivity that gets *reported* and one that gets *asserted*.

**Every gate in this Part is non-blocking, and that is a position rather than
caution.** Each measures either the consequence of a prior nothing identifies or a
property of a forward tail nothing has validated. A threshold on the first would
test only that the prior was applied; a threshold on the second would assert exactly
the thing that is unknown. They are recorded so a reader can see the number and
argue with it.

## §P0 — Replay configuration

`KalmanPortfolioConfig` is frozen like the other two. Its `results_path` defaults to
the **same v2 root**, so a replay lands beside the run it replays.

> **On the export root.** `get_export_state()` is a lazy singleton that resolves the
> root **once**, so `pviz.install` below does not move it — Part B's artifacts land
> in the same tree as Part A's, in their own section directories
> (`15c_forecast`, `15d_sweeps`, `15e_books`, `14b_recommendations`). Do not expect a
> second tree.

`rank_arms` is set to **all three** ranking rules. With one arm `plot_two_books` and
`plot_rank_agreement` are no-ops, and the disagreement between arms is the single
most informative thing this Part produces. **The first arm is the recommendation**
and the rest are contrasts — the export says so on the row (`book_role`), because a
reader handed two books from one posterior with no statement of precedence will use
whichever they find first.

`sector_cap` defaults to 0.30. That **moves** the shipped book, deliberately: no cap
is also a decision, and it should never be taken by omission.

In [ ]:
pcfg = kp.KalmanPortfolioConfig.from_env()
pcfg = replace(
    pcfg,
    rank_arms=tuple(RANKING_RULES),      # first = recommendation, rest = contrasts
    scenarios=run_cfg.forecast_scenarios,
    factor_share=run_cfg.forecast_factor_share,
    random_seed=run_cfg.random_seed,
    # max_names=None  -> breadth is SOLVED, not chosen; min_weight decides it
    min_weight=0.005,
    sector_cap=0.30,
    objective='log_growth',
)

# The replay owns gate names the fit does not; merge them so the report renders a
# rationale for every row rather than a blank column.
GATE_CATALOGUE.update(kp.PORTFOLIO_GATES)

pviz.install(pcfg, enable=EXPORT_ARTIFACTS)
report_p = GateReport()

print('handoff       :', pcfg.handoff)
print('exists        :', pcfg.handoff.exists())
print('results root  :', pcfg.results_path)
print('rank arms     :', pcfg.rank_arms, ' (first is the recommendation)')
print('objective     :', pcfg.objective, f'  (of {sorted(PORTFOLIO_OBJECTIVES)})')
print('breadth       : max_names=%s (ceiling)  min_weight=%.3f (decides it)'
      % (pcfg.max_names, pcfg.min_weight))
print('caps          : weight=%.2f sector=%s' % (pcfg.weight_cap, pcfg.sector_cap))
print('forecast      : %dd horizon / %dd steps / %d scenarios / factor_share=%.2f'
      % (pcfg.horizon_days, pcfg.step_days, pcfg.scenarios, pcfg.factor_share))

## §P0c — Load the handoff and the fit's frames

When Part A ran, the screen / MC summary / diagnostics are already in memory and are
reused. Otherwise they are read from the v2 results tree — `_read_v2_artifact` tries
the section directory first and the pre-migration flat path second, and **logs which
one it used**. That fallback is explicit on purpose: a silent degradation here costs
the replay its sector labels, its bounded ranking arm and its size-down watch, and
`_read_optional_csv` never fails.

The panel frame is read only for the analyst consensus that older screens lack; a
missing one costs `consensus_gap` and nothing else.

In [ ]:
handoff = kp.load_handoff(pcfg, report_p)
print(handoff.describe())

if FIT_MODEL:
    screen_p = screen
    mc_summary = frames.get('10_screen_mc_summary_v2')
    diagnostics_p = diagnostics.reset_index()
    panel_frame_p = panel.frame
    print('\nusing the in-memory frames from Part A')
else:
    screen_p = kp._read_v2_artifact(pcfg, '10_screen_results_v2')
    mc_summary = kp._read_v2_artifact(pcfg, '10_screen_mc_summary_v2')
    diagnostics_p = kp._read_v2_artifact(pcfg, '09_diagnostics_v2')
    panel_frame_p = kp._read_v2_artifact(pcfg, '04_panel_frame_v2')
    print('\nread from disk:', {
        'screen': None if screen_p is None else len(screen_p),
        'mc_summary': None if mc_summary is None else len(mc_summary),
        'diagnostics': None if diagnostics_p is None else len(diagnostics_p),
        'panel_frame': None if panel_frame_p is None else len(panel_frame_p),
    })

## §P1 — Forward simulation

Simulates joint forward returns from the handoff's posterior latent over the
configured horizon, then contrasts them against the shipped AR simulator **joined by
ISIN**.

Two summaries, and they answer different questions:

- **pooled** (`terminal=False`) — per-step marginals, directly comparable with the
  AR simulator's `er_*`.
- **terminal** (`terminal=True`) — the cumulative-horizon quantity, which is what
  the books are actually sized from.

`plot_engine_contrast` draws the second against the first with a y=x anchor. The
spread around the diagonal is the size of the decay-model choice, not noise, and the
Spearman in the title is computed on the **full** frame while the cloud is
decimated — a statistic annotated on a decimated panel that was computed on the
decimation is a different statistic.

In [ ]:
forecast = kp.run_forecast(handoff, pcfg, report_p, mc_summary=mc_summary)
fc = forecast['draws']
print(f'paths    : {fc.paths.shape}  (isin, scenario, step)')
print(f'terminal : {fc.terminal.shape}  <- the decision quantity')
print(f'backend  : {fc.backend}   factor_share={fc.factor_share}   '
      f'horizon={fc.horizon_days}d')

with pviz.section('15c_forecast'):
    pviz.plot_engine_contrast(forecast.get('engines'))

display(forecast['summary'].head())

## §P2 — The two prior sweeps

**This is the argument for the whole layer.**

`forecast_error_multiplier` is what buys the model's departure from analyst
consensus — at kappa = 0 the latent is unshrunk and the screen reproduces consensus
almost exactly (run `49e84d7e9d59` measured Spearman 0.999995). It is a **prior
chosen from a feasible band**, not an estimate, and the sweep shows how much of the
model's distinctiveness is a function of one chosen number.

`factor_share` is the second. Read its third panel carefully:
`er_sd_max_abs_diff` **should be flat at Monte-Carlo noise**, because the variance
split is preserving. A curve there would mean the invariance had broken and every
ratio built on `exp_vol` had quietly moved.

The multiplier sweep is an approximation of the shipped shrinkage — the same *shape*
(a pull toward the pooled mean) applied to the persisted latent — because what it is
for is the sensitivity, and the sensitivity is a property of the shape. Use
`scripts/profile_forecast_error.py` for the exact form.

In [ ]:
rank_values = None
if screen_p is not None and 'p_upside_pos_cond' in screen_p.columns:
    _keyed = (screen_p.drop_duplicates('isin')
              .set_index('isin')['p_upside_pos_cond'])
    rank_values = pd.to_numeric(
        _keyed.reindex(np.asarray(fc.isins)), errors='coerce').to_numpy()

sweeps = kp.run_prior_sweeps(
    handoff, pcfg, report_p,
    which=('factor_share', 'multiplier'),
    rank_values=rank_values,
)

with pviz.section('15d_sweeps'):
    pviz.plot_factor_sweep(sweeps.get('factor_share'), pcfg.factor_share)
    pviz.plot_multiplier_sweep(sweeps.get('multiplier'))

for _name, _frame in sweeps.items():
    print(f'--- {_name} ---')
    display(_frame)

## §P3 — Decision books

One book per ranking arm, on **one posterior**, so the disagreement between them is
attributable to the ranking rule and nothing else.

**Breadth is an output.** The ranking orders candidates and seeds the solve; it no
longer cuts at `k`. The solver runs over the eligible set, drops weights below
`min_weight`, and **re-solves** — repeatedly, because one pass is not enough: the
re-solve redistributes capital and a survivor can fall back under the floor.
`n_book` is *reported*, never chosen. Run `807df55e7158` published fifty names of
which thirty-eight held 1.17% between them and the smallest held 0.0002% — an
effective N of 11.7, which nothing reported; it took a human reading a bar chart.

Seven panels, and two of them are the suite's real argument:

- **`plot_denominator_sanity`** — a reward-to-risk ratio lets *thin evidence inflate
  a score*, so a book selected on it holds names whose modelled downside sits orders
  of magnitude below the universe median. Both shipped ratio candidates failed
  exactly here.
- **`plot_shrinkage_contrast`** (§P5) is its mirror: the one place in the pipeline
  where thin evidence pulls a signal *toward zero* instead. Same input condition,
  opposite treatment. Read them together.

`plot_risk_ladder` is a claim about **sign**: whether the 95%-worst modelled terminal
outcome is a gain for every name in the book. `plot_kelly_pin` says how often the
bisection never turned over — a pinned fraction means no draw loses money, which is a
statement about the simulation's left tail, not a sizing recommendation.

In [ ]:
decision = kp.run_decision_books(
    forecast, handoff, pcfg, report_p, screen=screen_p
)
books = decision.get('decision_frame')

for _arm, _bk in decision['books'].items():
    _s = _bk.summary
    print(f"[{_arm:20s}] n_book={int(_s['n_book']):3d}  "
          f"effective_N={_s['effective_n']:5.1f}  "
          f"E[r]={_s['port_expected']:+.2%}  "
          f"GVaR={_s['port_gvar']:+.2%}  "
          f"log_growth={_s['log_growth']:+.4f}")

with pviz.section('15e_books'):
    pviz.plot_two_books(books)
    pviz.plot_sector_mix(books, pcfg.sector_col, pcfg.sector_cap)
    pviz.plot_kelly_pin(books)
    pviz.plot_risk_ladder(books)
    pviz.plot_denominator_sanity(books)
    pviz.plot_rank_agreement(decision.get('agreement'))
    pviz.plot_ergodicity(decision.get('wealth_curve'))

display(decision.get('agreement'))

### §P3b — The mean-variance contrast

A **labelled contrast, never the recommendation.** Mean-variance treats volatility
as total risk — symmetric, so it charges a name for its upside — optimises one
period, and is famously unstable in the return estimates, which here are posterior
means of a latent that moves between refreshes. The book ships on expected log
growth; this says what the single-period answer would have been on the same draws.

Solved over the recommendation's **own holdings**, not the universe: the quadratic
objectives need a covariance and refuse above `MAX_SOLVER_NAMES = 400` (a 6.5k-name
covariance is 340 MB and will not converge), and the interesting question is what a
different objective does with the *same* candidates.

Note the solved frontier sits strictly outside the Dirichlet cloud that
`mean_variance_frontier` keeps for plotting — measured on one run: solved tangency
Sharpe 4.35 against the cloud's 0.69 on the same draws. **No annualisation**: the
draws are one NTM-horizon return and the textbook `* 252` would invent a frequency
they do not have.

In [ ]:
frontier = decision.get('frontier')
tangency = decision.get('tangency')
if frontier is not None:
    print(f"tangency Sharpe   : {tangency['sharpe']:.3f} "
          f"on {int(tangency['n_holdings'])} names")
    print(f"shipped book      : return {frontier['book_return'].iloc[0]:+.2%}  "
          f"vol {frontier['book_vol'].iloc[0]:.2%}  "
          f"log_growth {frontier['book_log_growth'].iloc[0]:+.4f}")
    display(frontier.head())
else:
    print('frontier contrast unavailable (see the log for the reason)')

## §P4 — Mean-model arms

**The one place this notebook can do something the CLI replay cannot.**

`kp.run_mean_model_arms` refuses from a replay and says why: a handoff carries the
four quantities the *simulator* reads, not the model graph, so a mean-structure
contrast has nothing to re-screen. Its warning tells the reader to run the stage from
the v2 workflow instead.

Here, when Part A ran, the live `panel` and `idata` are in scope — so this cell does
exactly that, calling `compare_arms_fast` directly. Max-and-Smooth regenerates the
latent under an arm in seconds against a **frozen covariance**, and that freeze is a
feature: it isolates the mean-structure effect instead of confounding it with a
refitted noise model.

In [ ]:
if FIT_MODEL:
    set_export_section('09b_comparison')

    # What run_mean_model_arms would delegate to, run against the LIVE panel.
    mean_arms = compare_arms_fast(
        panel, idata, model_cfg, run_cfg, report_p,
        arms=('hierarchy_nested', 'hierarchy_styled'),
    )
    display(mean_arms if mean_arms is not None else 'no arms screened')
else:
    kp.run_mean_model_arms(handoff, pcfg, report_p,
                           arms=('hierarchy_nested',))
    print('Replay mode: a handoff carries no panel or model graph, so this stage '
          'records its gate and defers. Re-run with FIT_MODEL = True.')

## §P5 — Recommendations

Turns the ranked frames into a **posture**: group over/underweights, per-name
actions, the size-down veto list and a reliability verdict.

The group signals run over the forecast's **terminal** returns — the quantity the
books are sized from — so the posture and the book answer the same question. (v1's
version runs over the upside posterior; both are legitimate and the two are a
contrast.)

`plot_shrinkage_contrast` is the panel to read beside §P3's denominator plot. Every
point lies between the y=x line and zero, and how far it falls short of y=x is
exactly `lambda_g` — thin evidence pulled *toward zero*, the opposite of what a
reward-per-risk ratio does with the same condition.

**Watch the action ladder.** The three-valued list this replaces returned 83.5% BUY
on run `807df55e7158` and nothing said so. Five rungs do not fix that by themselves:
the gates are scaled by the universe-mean confidence, so a low mean pulls the STRONG
threshold down onto what used to be the ordinary one and the top rung inherits the
same population under a stronger name. The gate positions are annotated on the panel
for exactly that reason.

In [ ]:
recommendations = kp.run_recommendations_v2(
    forecast, handoff, decision, pcfg,
    screen=screen_p, diagnostics=diagnostics_p, panel_frame=panel_frame_p,
    report=report_p, render=True,
)

with pviz.section('14b_recommendations'):
    _signals = recommendations.get('group_signals')
    pviz.plot_group_signal_forest(_signals)
    pviz.plot_shrinkage_contrast(_signals)
    pviz.plot_size_down_overlap(recommendations.get('watch'), books)
    _actions = recommendations.get('actions')
    pviz.plot_action_ladder(_actions)
    pviz.plot_consensus_gap(_actions)

## §P6 — Export & the replay gate report

CSV by default and by design. This workflow exists to be run many times over one
fit; a stage that wrote to the analytics schema on every run would not be that, and
the v2 tables are DROP-and-RECREATE — so a replay that wrote would destroy the export
it was replaying.

Each stem is filed through the shared layout SSOT into its own section directory, not
one `15_portfolio` bucket. That bucket was four stages under one folder name; the
stems already carried the section numbers, only the directories were missing.

In [ ]:
replay_frames = {
    '15c_forecast_summary': forecast.get('summary'),
    '15c_forecast_engines': forecast.get('engines'),
    '15d_factor_share_sweep': sweeps.get('factor_share'),
    '15d_multiplier_sweep': sweeps.get('multiplier'),
    '15e_decision_books': decision.get('decision_frame'),
    '15e_book_agreement': decision.get('agreement'),
    '15e_frontier': decision.get('frontier'),
    '14b_group_signals': recommendations.get('group_signals'),
    '14b_name_actions': recommendations.get('actions'),
    '14b_size_down_watch': recommendations.get('watch'),
    '09_gate_report_portfolio': report_p.to_frame(),
}

replay_counts = kp.export_frames(
    {k: v for k, v in replay_frames.items() if v is not None and len(v)},
    pcfg,
    get_export_state().run_id,
    identity=screen_p if screen_p is not None else handoff.identity,
) if EXPORT_ARTIFACTS else {}

for _k, _v in sorted(replay_counts.items()):
    print(f'{_k:30s}: {_v:>7,} rows')

print(report_p.render())

---

## Where the artifacts landed

Both Parts write into the **v2 results root** (`KALMAN_V2_RESULTS_DIR`, default
`pymc_kalman_filter_pt_v2_results/`), one directory per stage. v1 has its own tree
under `KALMAN_PT_RESULTS_DIR`; the separation is load-bearing — the two shared a
variable until 2026-08-27, and a v2 run scattered its frames through v1's tree under
names one suffix from v1's own.

| Part | Directories |
|---|---|
| A | `01_data`, `03_features`, `04_panel`, `04b_audit`, `06_prior`, `07_posterior`, `08_ppc`, `09_diagnostics`, `09b_comparison`, `09_gates`, `10_screen`, `10b_risk`, `10c_analytics`, `15_forecast`, `15b_decision` |
| B | `15c_forecast`, `15d_sweeps`, `15e_books`, `14b_recommendations`, `09_gates` |

Never build a result path by hand — `export_layout.section_path(root, stem)` is the
only sanctioned stem-to-path conversion, and a stem written to the root is
indistinguishable from a stray file. Re-file an older flat tree with:

```powershell
python pymc_kalman_filter_pt_v2.py --migrate-layout            # dry run, lists moves
python pymc_kalman_filter_pt_v2.py --migrate-layout --apply
```

## One-shot CLI equivalents

```powershell
$env:PYTHONIOENCODING = "utf-8"     # in the SHELL; setting it in-script is too late

python pymc_kalman_filter_pt_v2.py --dry-run     # §4b only, seconds
python pymc_kalman_filter_pt_v2.py               # full fit + export
python kalman_portfolio.py --rank-arms all --sweep factor_share,multiplier
python kalman_portfolio.py --fit                 # fit first, then replay
```

For a **production** analytics refresh use `scripts/export_kalman_analytics.py`
rather than `WRITE_ANALYTICS = True` here — and treat the export and the GEIB
dashboard deploy as a pair, since the dashboard reads the table this would replace.

## Before trusting an export

- Read the **gate report**, not the absence of a traceback. `export_finite` is
  blocking; the rest report.
- Check the run is one vintage: every frame carries `run_id`, `exported_at`,
  `source_sha`, `source_dirty`. A mixed-vintage schema is invisible at the schema
  level — it once served two fits at once and took a value-level diff to find.
- `source_dirty = true` is a fact, not an error.

## Saving this notebook

Clear outputs before committing. Plotly serialises every coordinate it draws into
the `.ipynb`, and a v1 notebook once reached **233 MB, 207.7 MB of it in a single
prior-predictive figure**. The panels here go through the shared payload
primitives — binned densities, gridded ECDFs, decimated scatters with the sampled
count in the title — so a large file means outputs were saved, not that a figure
misbehaved.